In [3]:
import json
import numpy as np
from collections import defaultdict
from datetime import datetime, timezone
from astropy.time import Time
from astropy.coordinates import EarthLocation
import astropy.units as u

In [ ]:
def unix_to_lst(unix_t, coords):
    loc = EarthLocation(lon=coords[1]*u.deg, lat=coords[0]*u.deg, height=coords[2]*u.m)
    t_obj = Time(unix_t, format="unix", location=loc)
    lst_time = t_obj.sidereal_time("mean").degree % 360
    return lst_time


def get_sim_passes_lst(json_paths, coords, same_sat = False):
    """
    get the simultaneous passes across pulse_data files for same lst times

    Parameters
    ----------
    json_paths : list of str
        paths to the json pulse_data files. 
    location : astropy.coordinates.EarthLocation
        where the telescope is

    Returns
    -------
    sim_passes: list of dicts
        list of all the passes which happen in two or more files at same time
    """

    # STEP 1: extract all pulses into a dictionary
    interval_bounds = []  
    pulse_dict = {} 

    for path in json_paths:
        with open(path, "r") as f:
            data = json.load(f)

        for interval_str, antenna_data in data.items():
            interval_start = int(interval_str)
            interval_end = interval_start
            for ant_key, ant_dict in antenna_data.items():
                for pulse in ant_dict["pulse_data"]:
                    unix_start = int(interval_start+ pulse["start"])
                    unix_end = int(interval_start + pulse["end"])
                    lst_start = unix_to_lst(unix_start, coords)
                    lst_end = unix_to_lst(unix_end, coords)

                    if unix_end >interval_end:
                        interval_end = int(unix_end)

                    for sat_id in pulse["sats_present"]:
                        if unix_start not in pulse_dict:
                            pulse_dict[unix_start] = {
                                "lst": (lst_start, lst_end),
                                "unix": (unix_start, unix_end),
                                "antenna": set(),
                                "sats": set(),
                                "file": str(path),
                            }
                        pulse_dict[unix_start]["antenna"].add(ant_key)
                        pulse_dict[unix_start]["sats"].add(sat_id)

            interval_bounds.append((interval_start, interval_end, path))


    # STEP 2: CHECK NO FILES OVERLAP
    sorted_bounds = sorted(interval_bounds, key=lambda x: x[0])
    for i in range(1, len(sorted_bounds)):
        prev_end = sorted_bounds[i-1][1]
        curr_start = sorted_bounds[i][0]
        if curr_start < prev_end:
            raise ValueError(f"Intervals overlap: {sorted_bounds[i-1]} and {sorted_bounds[i]}")


    #STEP 3: CONVERT PULSE DICTIONARY TO LIST AND SORT
    pulse_entries = []
    for entry in pulse_dict.values():
        entry["antenna"] = list(entry["antenna"])
        entry["sats"] = list(entry["sats"])
        pulse_entries.append(entry)
        
    pulse_entries.sort(key=lambda x: x['lst'][0])  # Sort by LST start

    #STEP 4: GET OVERLAPPING PULSES
    sim_pulses = []
    for i in range(len(pulse_entries)):
        e1 = pulse_entries[i]
        lst1_start, lst1_end = e1['lst']
        group = [e1]

        for j in range(i+1, len(pulse_entries)):
            e2 = pulse_entries[j]
            lst2_start, lst2_end = e2['lst']

            if lst2_start >= lst1_end:
                break  # can't overlap any more, go to next

            if same_sat:
                if not set(e1["sats"]) & set(e2["sats"]):  # no common satellite
                    continue

            if lst2_start < lst1_end and lst2_end > lst1_start:
                group.append(e2)

        if len(group) >= 2:
            sim_pulses.append(group)

    return sim_pulses

In [29]:
paths = [
    '/scratch/thomasb/pulsedata_1753132820_len_67200_1760024247.5361912.json',
    '/scratch/thomasb/pulsedata_1753270395_len_9000_1760804388.json'
]


In [35]:
data = get_sim_passes_lst(paths, [79.41718333333333, -90.76735, 189], same_sat=False)

In [36]:
for group in data:
    print(group)

[{'lst': (54.317813494030396, 55.55034551873576), 'unix': (1753277650, 1753277945), 'antenna': ['Antenna 8', 'Antenna 2'], 'sats': ['57166'], 'file': '/scratch/thomasb/pulsedata_1753270395_len_9000_1760804388.json'}, {'lst': (54.89894064772783, 57.13421059397844), 'unix': (1753191625, 1753192160), 'antenna': ['Antenna 7', 'Antenna 4', 'Antenna 2'], 'sats': ['59051'], 'file': '/scratch/thomasb/pulsedata_1753132820_len_67200_1760024247.5361912.json'}]
[{'lst': (59.13968643360689, 60.455779953359084), 'unix': (1753192640, 1753192955), 'antenna': ['Antenna 7', 'Antenna 8', 'Antenna 6'], 'sats': ['57166'], 'file': '/scratch/thomasb/pulsedata_1753132820_len_67200_1760024247.5361912.json'}, {'lst': (60.125337271457134, 61.60855377576142), 'unix': (1753279040, 1753279395), 'antenna': ['Antenna 7', 'Antenna 4', 'Antenna 2', 'Antenna 8', 'Antenna 6'], 'sats': ['25338'], 'file': '/scratch/thomasb/pulsedata_1753270395_len_9000_1760804388.json'}]
[{'lst': (60.125337271457134, 61.60855377576142), 'u